## Limpeza dos dados: raw para cleaned
Esse notebook lê as 20 tabelas da camada `raw`, aplica só os tratamentos que os erros mapeados no dataload pedem (nulo escrito como texto e chave nula) e salva o resultado na camada `cleaned`.

#### Rotina de conexão com o Neon

In [0]:
%pip install python-dotenv

In [0]:
dbutils.library.restartPython()

In [0]:
%run ../apollo-core/apollo_connection_neon_core

#### Visualizar e analisar os dados da base

##### Imports

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StringType

##### Colunas de texto de uma tabela

In [0]:
def text_columns(df):
    return [f.name for f in df.schema.fields if isinstance(f.dataType, StringType)]

##### Perfil de uma tabela

In [0]:
def profile(df):
    types = dict(df.dtypes)
    rows = []
    for column in df.columns:
        nulls = df.filter(F.col(column).isNull()).count()
        distinct = df.select(column).distinct().count()
        rows.append((column, types[column], nulls, distinct))
    return spark.createDataFrame(rows, "coluna string, tipo string, nulos long, distintos long")

##### Rodar o perfil em uma tabela

In [0]:
df = spark.table("workspace.raw.dim_company")
display(profile(df))

##### Olhar as categorias dos textos

In [0]:
for column in text_columns(df):
    if df.select(column).distinct().count() <= 15:
        display(df.groupBy(column).count().orderBy("count", ascending=False))

#### Tratamento dos dados

##### Nulo escrito como texto

In [0]:
FAKE_NULLS = ["s/n"]

def fake_null_to_null(df):
    for column in text_columns(df):
        df = df.withColumn(column, F.when(F.lower(F.col(column)).isin(FAKE_NULLS), None).otherwise(F.col(column)))
    return df

##### Chave nula

In [0]:
def drop_null_keys(df, keys):
    return df.dropna(subset=keys)

#### Chaves de cada tabela

##### Chaves por tabela

In [0]:
# tabela sem chave listada só passa pelo tratamento de nulo escrito como texto
TABLE_KEYS = {
    "dim_company": ["company_key"],
    "dim_segment": ["segment_key"],
    "dim_company_unit": ["company_unit_key"],
    "dim_panel_model": ["panel_model_key"],
    "dim_inverter": ["inverter_key"],
    "dim_string": ["string_key"],
    "dim_panel": ["panel_key"],
    "dim_employee": ["employee_key"],
    "dim_date": ["date_key"],
    "dim_maintenance_type": ["maintenance_type_key"],
    "dim_part": ["part_key"],
    "fact_string_daily_performance": ["string_key", "date_key"],
    "fact_string_health": ["string_key", "calculated_at"],
    "fact_maintenance": ["maintenance_key"],
    "fact_part_consumption": ["maintenance_key", "part_key"],
    "fact_daily_active_user": ["company_key", "date_key"],
    "fact_warning": ["warning_key"],
    "fact_relocation": ["relocation_scope", "relocation_key"],
    "fact_stock": ["company_unit_key", "part_key"],
    "vw_maintenance_chain": ["id"],
}

#### Salvar as tabelas na cleaned

##### Tratar uma tabela

In [0]:
def clean_table(df, keys):
    df = fake_null_to_null(df)
    return drop_null_keys(df, keys) if keys else df

##### Criar o schema da cleaned

In [0]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.cleaned")

##### Rodar o tratamento e salvar cada tabela

In [0]:
for table in VIEWS:
    raw_df = spark.table(f"{CATALOG}.raw.{table}")
    cleaned_df = clean_table(raw_df, TABLE_KEYS.get(table, []))
    cleaned_df.write.mode("overwrite").saveAsTable(f"{CATALOG}.cleaned.{table}")
    print(table, "ok:", raw_df.count() - cleaned_df.count(), "linhas removidas")

#### Conferir as tabelas

In [0]:
display(spark.sql(f"SHOW TABLES IN {CATALOG}.cleaned"))